### Import

In [1]:
%%capture
import os, importlib.util
%pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
import torch
if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
    !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
else:
    os.environ["FLA_TILELANG"] = "0"
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [2]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
import torch
import json
from pathlib import Path
import torch
print(torch.cuda.is_available())
from unsloth import FastVisionModel # FastLanguageModel for LLMs



c:\Users\yujie.lim\OneDrive - SK Jewellery\VSCode Projects\Helpful\Data_Ingest\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [7]:
fourbit_models = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = True, # Use 4bit to reduce memory use. False for 16bit LoRA.
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

NameError: name 'FastVisionModel' is not defined

In [1]:
file_path = Path("annotations.json")
writeAnnotation = 0

if writeAnnotation == 1:
    annotations = {}
    for i in range(86):
        annotations[f"myKad{i}.jpg"] = {
            "fields": {
                "Id-no": "",
                "Name": "",
                "Address1": "",
                "Address2": "",
                "Address3": "",
                "Postal_Code": "",
                "State": "",
                "Gender": "",
            }
        }
    
    with open("annotations.json", 'w') as f:
        json.dump(annotations, f, indent=2)
    
    print(f"Generated annotations.json with {len(annotations)} entries")

NameError: name 'Path' is not defined

### Train-test-split

In [3]:
# Assuming you have your image-text pairs
# images: list of PIL Images or image paths
# texts: list of captions/prompts

# Create a dataset dict
data = {
    'image': "myKad0.jpg",
    'text': {
      "Id-no": "030612-14-0440",
      "Name": "Yong Jia Yi",
      "Address1": "17 JALAN 1/2A",
      "Address2": "BANDAR DAMAI PERDANA",
      "Address3": "CHERAS",
      "Postal_Code": "56000 KUALA LUMPUR",
      "State": "W.PERSEKUTAN(KL)",
      "Gender": "PEREMPUAN"
    }
}
dataset = Dataset.from_dict(data)

# Split it - 80/20 is standard, adjust if needed
train_test = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test['train']
test_dataset = train_test['test']

# For QLoRA fine-tuning with Qwen-VL, you'll need a collate function
def collate_fn(batch):
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    # Process based on your model's processor
    return {'images': images, 'texts': texts}

# Then pass to DataLoader
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, collate_fn=collate_fn)

ArrowInvalid: Column 1 named text expected length 10 but got length 8